In [1]:
from dotenv import load_dotenv

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser,JsonOutputParser


load_dotenv()

True

In [2]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are an AI Professor."),
        ("human","Explain {topic} in one paragraph.")
    ]
)

In [4]:
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke(
    {
        "topic":"Transformers"
    }
)

print(response)

Transformers are a type of neural network architecture introduced in 2017, revolutionizing the field of natural language processing (NLP). They're primarily designed for sequence-to-sequence tasks, such as machine translation, text summarization, and chatbots. The key innovation of Transformers is their reliance on self-attention mechanisms, which allow the model to weigh the importance of different input elements relative to each other, rather than relying on recurrent neural networks (RNNs) or convolutional neural networks (CNNs). This enables Transformers to handle long-range dependencies and parallelize computations more efficiently, making them particularly well-suited for tasks that involve complex sequential data. The Transformer architecture consists of an encoder and a decoder, with the encoder generating a continuous representation of the input sequence, and the decoder generating the output sequence, one element at a time, based on the encoder's output and self-attention mec

##### JsonOutputParser

In [5]:
parser = JsonOutputParser()

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Return the answer in valid JSON format.

            {{
                "name":"",
                "age":"",
                "city":""
            }}
            """
        ),
        (
            "human",
            "Generate details for a student."
        )
    ]
)

In [7]:
chain = prompt | llm | parser

response = chain.invoke({})

response

{'name': 'John Doe', 'age': '20', 'city': 'New York'}

##### PydanticOutputParser

In [9]:
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser


In [10]:
class Student(BaseModel):

    name:str = Field(
        description="Student name"
    )

    age:int = Field(
        description="Student age"
    )

    city:str = Field(
        description="Student city"
    )

In [11]:
parser = PydanticOutputParser(
    pydantic_object=Student
)

In [12]:
format_instructions = parser.get_format_instructions()

print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "Student name", "title": "Name", "type": "string"}, "age": {"description": "Student age", "title": "Age", "type": "integer"}, "city": {"description": "Student city", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}
```


In [13]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            {format_instructions}
            """
        ),
        (
            "human",
            "Generate details of a student."
        )
    ]
)

In [14]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "format_instructions":
        format_instructions
    }
)

response

Student(name='John Doe', age=20, city='New York')